In [3]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("../cms_hospital.db")
paradox_hospitals = pd.read_csv('../outputs/paradox_hospitals.csv')
paradox_hospitals['facility_id'] = paradox_hospitals['facility_id'].astype(str).str.zfill(6)

print(f"Loaded {len(paradox_hospitals)} paradox hospitals")

Loaded 99 paradox hospitals


#### Layer 4 — The patient experience disconnect

*Do patients in high-spending hospitals rate them higher? Or are those hospitals spending more and still delivering poor satisfaction?*

In [4]:
hcahps_comparison = pd.read_sql_query("""
    WITH paradox_hcahps AS (
        SELECT
            'Paradox Hospitals' AS group_name,
            ROUND(AVG(H_STAR_RATING), 2)              AS avg_star_rating,
            ROUND(AVG(H_HSP_RATING_LINEAR_SCORE), 2)  AS avg_overall_hospital_rating,
            ROUND(AVG(H_RECMND_LINEAR_SCORE), 2)      AS avg_recommend_score,
            ROUND(AVG(H_COMP_1_LINEAR_SCORE), 2)      AS avg_nurse_communication,
            ROUND(AVG(H_COMP_2_LINEAR_SCORE), 2)      AS avg_doctor_communication,
            ROUND(AVG(H_COMP_5_LINEAR_SCORE), 2)      AS avg_discharge_info,
            ROUND(AVG(H_COMP_6_LINEAR_SCORE), 2)      AS avg_care_transition,
            ROUND(AVG(H_CLEAN_LINEAR_SCORE), 2)       AS avg_cleanliness,
            ROUND(AVG(H_QUIET_LINEAR_SCORE), 2)       AS avg_quietness
        FROM hcahps_patient_survey
        WHERE facility_id IN ({})
    ),
    non_paradox_hcahps AS (
        SELECT
            'All Other Hospitals' AS group_name,
            ROUND(AVG(H_STAR_RATING), 2)              AS avg_star_rating,
            ROUND(AVG(H_HSP_RATING_LINEAR_SCORE), 2)  AS avg_overall_hospital_rating,
            ROUND(AVG(H_RECMND_LINEAR_SCORE), 2)      AS avg_recommend_score,
            ROUND(AVG(H_COMP_1_LINEAR_SCORE), 2)      AS avg_nurse_communication,
            ROUND(AVG(H_COMP_2_LINEAR_SCORE), 2)      AS avg_doctor_communication,
            ROUND(AVG(H_COMP_5_LINEAR_SCORE), 2)      AS avg_discharge_info,
            ROUND(AVG(H_COMP_6_LINEAR_SCORE), 2)      AS avg_care_transition,
            ROUND(AVG(H_CLEAN_LINEAR_SCORE), 2)       AS avg_cleanliness,
            ROUND(AVG(H_QUIET_LINEAR_SCORE), 2)       AS avg_quietness
        FROM hcahps_patient_survey
        WHERE facility_id NOT IN ({})
    )
    SELECT * FROM paradox_hcahps
    UNION ALL
    SELECT * FROM non_paradox_hcahps
""".format(
    ','.join(['?']*len(paradox_hospitals)),
    ','.join(['?']*len(paradox_hospitals))
), conn, params=paradox_hospitals['facility_id'].tolist()*2)

hcahps_comparison

,group_name,avg_star_rating,avg_overall_hospital_rating,avg_recommend_score,avg_nurse_communication,avg_doctor_communication,avg_discharge_info,avg_care_transition,avg_cleanliness,avg_quietness
0,Paradox Hospitals,2.42,84.78,82.90,88.84,88.71,73.06,82.72,83.58,78.38
1,All Other Hospitals,3.27,88.03,87.06,91.28,90.78,76.58,85.84,86.70,81.90


### Layer 4 — Patient Experience: Patients Feel It Too

Do patients in paradox hospitals rate them differently?
The HCAHPS scores say yes — not dramatically, but consistently.

| Dimension | Paradox Hospitals | All Other Hospitals | Gap |
|-----------|------------------|---------------------|-----|
| Star Rating | 2.42 | 3.27 | -0.85 |
| Overall Hospital Rating | 84.78 | 88.03 | -3.25 |
| Recommend Score | 82.90 | 87.06 | -4.16 |
| Nurse Communication | 88.84 | 91.28 | -2.44 |
| Doctor Communication | 88.71 | 90.78 | -2.07 |
| Discharge Information | 73.06 | 76.58 | -3.52 |
| Care Transition | 82.72 | 85.84 | -3.12 |

**What the scores tell us:**

Patients are not outraged — they rate their doctors and nurses
relatively well. The human interaction inside the hospital
feels acceptable to most patients.

But the overall picture is lower across every single dimension.
No category where paradox hospitals match or beat their peers.
The dysfunction is felt, even if patients can't name it.

**The most telling gap is discharge information at -3.52.**
Patients leaving paradox hospitals feel less informed
about their care transition — less prepared for what comes next.
This is not a coincidence.
It directly mirrors the clinical finding from Layer 3:
these hospitals are poor at discharge planning,
and their patients experience that failure firsthand.

**The star rating gap of 0.85 is significant.**
On a 1-5 scale, paradox hospitals average 2.42 —
closer to 2 stars than 3.
Patients are quietly signaling that something is not right,
even when they cannot point to exactly what.

#### Layer 4 — Summary

Patients in paradox hospitals are not completely dissatisfied —
they appreciate their caregivers.
But they consistently rate every dimension of their experience lower
than patients in all other hospitals.

The disconnect is not that patients love these hospitals despite the poor outcomes.
The disconnect is subtler:
patients feel something is off, rate it slightly lower,
but don't have the clinical data to understand why.

The data does.